# Destination Prioritisation – BI analysis notebook

**Stakeholder:** Commercial Manager (primary), Finance Controller (secondary)
**Main question:** Which brands and destinations should the commercial team prioritise or review, based on demand, cancellations, sales and margin from 2020–2022?

| Section | Purpose | Course goal |
|---|---|---|
| 1. Setup and load | Load all six files | – |
| 2. Exploration | What the data contains | S16 |
| 3. Data quality checks | Nulls, duplicates, keys, dates, amounts, outliers | S16 (accuracy, limitations) |
| 4. Preparation decisions | What we label, flag or keep – nothing is deleted | S16 |
| 5. Joins and master data | Master table + booking-level table + Power BI table | S17 (workflow) |
| 6. Reconciliation | Prove nothing was lost or duplicated | S16 / S17 (trust) |
| 7. Export | `booking_level_powerbi.csv` (48 columns) for Power BI, plus the check log | S17 |
| 8. Findings | Q1 demand, Q2 cancellations, Q3 performance and channels, decision rule | S16 |
| 9. From output to finding | Observation → interpretation → implication → action | S16 |

> **Data in this repository is synthetic.** Run `generate_dummy_data.py` first; it creates the six input files in `data/` with the same structure as the original (confidential) data.


# 1. Setup and load

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display

pd.set_option("display.float_format", "{:,.2f}".format)
pd.set_option("display.max_columns", 50)

DATA = Path("data")           # folder with the six input CSV files (change if needed)
OUT = Path("output")          # all exported files go here
OUT.mkdir(exist_ok=True)

# ---- Parameters (agreed by the team; change here only) ----
YEAR = 2022                   # default analysis year, by first_departure_date
MIN_BOOKINGS = 30             # minimum kept bookings before a destination is classified
HOME_CITIES = []              # fill in after check 3.4 if home cities appear, e.g. ["Stockholm"]
KEPT, CANC, PRELIM = "Booking", "Cancellation", "Preliminary"

files = ["fact_sales_and_costs", "dim_booking", "dim_passenger",
         "dim_transport", "dim_market", "dim_product_group"]
raw = {name: pd.read_csv(DATA / f"{name}.csv") for name in files}
fact, bk, pax, tr, mk, pg = (raw[n] for n in files)

# 2. Exploration
### 2.1 Table overview

In [ ]:
pd.DataFrame([{"table": n, "rows": len(df), "columns": df.shape[1]} for n, df in raw.items()])

### 2.2 Column profile per table
Data type, missing values, number of unique values and an example value for every column.

In [ ]:
def profile(df):
    return pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "nulls": df.isna().sum(),
        "null_%": (df.isna().mean() * 100).round(2),
        "unique": df.nunique(),
        "example": df.apply(lambda s: s.dropna().iloc[0] if s.notna().any() else None),
    })

for name, df in raw.items():
    print(f"\n=== {name} ===")
    display(profile(df))

### 2.3 Convert dates
Dates are stored as text. `errors='coerce'` turns unreadable values into missing, and the check below counts any that failed.

In [ ]:
date_cols = ["booking_date", "cancellation_date", "first_departure_date", "last_return_date"]
for c in date_cols:
    before = fact[c].notna().sum()
    fact[c] = pd.to_datetime(fact[c], errors="coerce")
    failed = before - fact[c].notna().sum()
    print(f"{c:22} min {fact[c].min().date()}  max {fact[c].max().date()}  failed to parse: {failed}")

### 2.4 Categories

In [ ]:
for col in ["booking_status", "sales_platform", "brand"]:
    print(f"\n{col} (bookings):")
    print(bk[col].value_counts(dropna=False))
print("\nsell_country / sell_currency:")
print(mk[["sell_country", "sell_currency"]].value_counts())
print("\ncreated_by – top 10:")
print(bk.created_by.value_counts(dropna=False).head(10))

### 2.5 Numeric distributions

In [ ]:
display(fact[["sales_global", "cost_global"]].describe())
display(pax["pax"].describe())
print(f"Rows with zero sales: {(fact.sales_global == 0).mean():.1%}")

# 3. Data quality checks
Every check adds a line to `quality_log` (check, result, decision). This log becomes the check record in the group note.

In [ ]:
quality_log = []
def log(area, check, count, decision):
    quality_log.append({"area": area, "check": check, "issue_count": int(count), "decision": decision})
    print(f"[{area}] {check}: {count}")

### 3.1 Duplicates

In [ ]:
for name, df in raw.items():
    log("Duplicates", f"{name}: fully duplicated rows", df.duplicated().sum(),
        "Investigate if > 0; not removed automatically")

keys = {"dim_booking": "booking_id", "dim_passenger": "passenger_id", "dim_transport": "transport_id",
        "dim_market": "market_id", "dim_product_group": "product_group_id"}
for name, key in keys.items():
    log("Duplicates", f"{name}: duplicate {key}", raw[name][key].duplicated().sum(),
        "Must be 0, otherwise joins duplicate fact rows")

log("Duplicates", "fact: repeated booking_id × product_group_id",
    fact.duplicated(["booking_id", "product_group_id"]).sum(),
    "Expected grain check; repeats may be legitimate (e.g. several lines per product group)")

### 3.2 Missing values in key fields

In [ ]:
for col in ["booking_id", "passenger_id", "market_id", "product_group_id", "transport_id",
            "first_departure_date", "sales_global", "cost_global", "last_return_date"]:
    log("Missing", f"fact.{col} missing", fact[col].isna().sum(),
        "Keep rows; label or flag (see section 4)")
log("Missing", "dim_booking.brand missing (bookings)", bk.brand.isna().sum(), "Label 'Unknown brand'")
log("Missing", "dim_product_group.description_english missing", pg.description_english.isna().sum(),
    "Label 'Unknown'")

### 3.3 Referential integrity
Does every key in the fact table exist in its dimension table?

In [ ]:
for col, (name, key) in {"booking_id": ("dim_booking", "booking_id"),
                         "passenger_id": ("dim_passenger", "passenger_id"),
                         "market_id": ("dim_market", "market_id"),
                         "product_group_id": ("dim_product_group", "product_group_id"),
                         "transport_id": ("dim_transport", "transport_id")}.items():
    orphans = fact[col].notna() & ~fact[col].isin(raw[name][key])
    log("Keys", f"fact.{col} not found in {name}", orphans.sum(), "Must be 0; investigate otherwise")

### 3.4 Destinations
Look for **home cities** (Stockholm, Copenhagen, Oslo, Helsinki…). If they appear, add them to `HOME_CITIES` in section 1 and re-run.

In [ ]:
f0 = fact.merge(tr[["transport_id", "arrival_city"]], on="transport_id", how="left")
display(f0.groupby("arrival_city", dropna=False)["booking_id"].nunique()
          .sort_values(ascending=False).head(20).rename("bookings"))
multi = f0[~f0.arrival_city.isin(HOME_CITIES)].dropna(subset=["arrival_city"]) \
          .groupby("booking_id")["arrival_city"].nunique()
log("Destination", "bookings with > 1 destination", (multi > 1).sum(),
    "Destination assigned once per booking (most frequent arrival_city)")
log("Destination", "rows with no destination", f0.arrival_city.isna().sum(),
    "Keep in totals as 'Unknown destination', drillable")

### 3.5 Booking-level checks
Status, dates and amounts are checked per booking, because a booking is what someone would investigate.

In [ ]:
b = (fact.merge(bk[["booking_id", "booking_status"]], on="booking_id", how="left")
         .groupby("booking_id")
         .agg(status=("booking_status", "first"),
              has_cancel_date=("cancellation_date", lambda s: s.notna().any()),
              booking_date=("booking_date", "min"),
              cancellation_date=("cancellation_date", "min"),
              departure=("first_departure_date", "min"),
              return_date=("last_return_date", "max"),
              sales=("sales_global", "sum"),
              cost=("cost_global", "sum"),
              missing_cost_rows=("cost_global", lambda s: s.isna().sum()),
              n_markets=("market_id", "nunique")))

b["flag_status_vs_cancel_date"] = (b.status == CANC) != b.has_cancel_date
b["flag_booked_after_departure"] = b.booking_date > b.departure
b["flag_cancelled_before_booking"] = b.cancellation_date < b.booking_date
b["flag_return_before_departure"] = b.return_date < b.departure
b["flag_kept_negative_sales"] = (b.status == KEPT) & (b.sales < 0)
b["flag_cancelled_positive_sales"] = (b.status == CANC) & (b.sales > 0)
b["flag_zero_sales"] = b.sales == 0
b["flag_missing_cost"] = b.missing_cost_rows > 0
b["flag_kept_loss"] = (b.status == KEPT) & (b.sales - b.cost < 0)
b["flag_preliminary"] = b.status == PRELIM
b["flag_multi_market"] = b.n_markets > 1

decisions = {
    "flag_status_vs_cancel_date": "Should be 0 (team check: status always agrees)",
    "flag_booked_after_departure": "Keep; flag for review (booking_date may be row-level)",
    "flag_cancelled_before_booking": "Keep; flag for review",
    "flag_return_before_departure": "Keep; flag for review",
    "flag_kept_negative_sales": "Keep; flag – unexpected for an active booking",
    "flag_cancelled_positive_sales": "Keep; flag – possibly cancellation fees",
    "flag_zero_sales": "Keep; flag",
    "flag_missing_cost": "Keep; margin overstated on these – disclose share",
    "flag_kept_loss": "Keep; candidate 'unexpected loss' for review",
    "flag_preliminary": "Show separately; should have auto-cancelled",
    "flag_multi_market": "Check how sell country is assigned",
}
for col, dec in decisions.items():
    log("Booking", col.replace("flag_", ""), b[col].sum(), dec)

### 3.6 Cancelled bookings: original sale + reversal, or reversal only?

In [ ]:
canc_rows = fact[fact.booking_id.isin(b.index[b.status == CANC])]
pos = canc_rows[canc_rows.sales_global > 0].booking_id.nunique()
log("Cancellation", "cancelled bookings with any positive sales row", pos,
    "If small vs all cancelled bookings: data holds the reversal, not the original sale")
print(f"of {int((b.status == CANC).sum())} cancelled bookings")

In [ ]:
canc_ids = b.index[b.status == CANC]
pos_rows = fact[fact.booking_id.isin(canc_ids) & (fact.sales_global > 0)]
(pos_rows.groupby("product_group_id").agg(rows=("sales_global", "size"),
                                          sales=("sales_global", "sum"))
 .reset_index()
 .merge(pg[["product_group_id", "description_english"]], how="left")
 .sort_values("sales", ascending=False).head(10))

### 3.7 Outliers

In [ ]:
pax_b = fact.drop_duplicates("booking_id")[["booking_id", "passenger_id"]].merge(pax, on="passenger_id", how="left")
print(pax_b.pax.describe())
log("Outlier", "bookings with pax >= 20", (pax_b.pax >= 20).sum(), "Keep; likely group bookings – verify")

kept_sales = b.loc[b.status == KEPT, "sales"]
q1, q3 = kept_sales.quantile([.25, .75])
upper = q3 + 3 * (q3 - q1)
log("Outlier", f"kept bookings with sales > {upper:,.0f} SEK (Q3 + 3×IQR)", (kept_sales > upper).sum(),
    "Keep; check they are not errors")

### 3.8 Brand vs sell country, positive cancellations, missing-cost share
If a brand sells almost only in one country, brand comparisons are partly country comparisons.

In [ ]:
sell_country_b = fact.groupby("booking_id").market_id.first().map(mk.set_index("market_id").sell_country)
display(pd.crosstab(bk.set_index("booking_id").brand.fillna("Unknown brand"),
                    sell_country_b.fillna("Unknown"), dropna=False))

print("Cancelled bookings with positive totals – sales by product group (cancellation fees?):")
pos_ids = b.index[b.flag_cancelled_positive_sales]
display(fact[fact.booking_id.isin(pos_ids)].groupby("product_group_id").sales_global.sum()
        .reset_index().merge(pg[["product_group_id", "description_english"]], how="left")
        .sort_values("sales_global", ascending=False))

k = fact[fact.booking_id.isin(b.index[b.status == KEPT])]
share = k.loc[k.cost_global.isna(), "sales_global"].sum() / k.sales_global.sum()
log("Missing", f"share of kept sales on rows with missing cost: {share:.2%}", k.cost_global.isna().sum(),
    "Disclose; margin % slightly overstated on these")

pre = b[b.status == PRELIM]
if len(pre):
    print(f"Preliminary bookings: {len(pre)}, booking dates {pre.booking_date.min().date()} to {pre.booking_date.max().date()}"
          " – all older than 24 h means they were not auto-cancelled")

### 3.9 Quality log

In [ ]:
quality_log_df = pd.DataFrame(quality_log)
quality_log_df

# 4. Preparation decisions

**We do not delete rows.** The stakeholder asked us to keep unknown and missing values in the totals, and deleting rows would break reconciliation with the source.

| Issue | Decision |
|---|---|
| Missing destination | Label `Unknown destination`, keep in totals, drillable |
| Missing brand | Label `Unknown brand` |
| Product group without description | Label `Unknown` |
| Missing `cost_global` | Keep as missing (counts as 0 in sums), flag, disclose share of sales |
| Preliminary bookings | Excluded from kept/cancelled measures, shown separately |
| Dates | Converted from text to dates |
| Period | `first_departure_date` |
| Currency | `sales_global` / `cost_global` in SEK; local amounts never summed |

In [ ]:
bk_c = bk.copy()
bk_c["brand"] = bk_c.brand.fillna("Unknown brand")
pg_c = pg.copy()
pg_c["description_english"] = pg_c.description_english.fillna("Unknown")

# 5. Joins and master data
### 5.1 Destination per booking
Destination = most frequent `arrival_city` on the booking's rows, excluding `HOME_CITIES` (ties: alphabetical). **This is our assumption** – document it.

In [ ]:
dest = (f0[~f0.arrival_city.isin(HOME_CITIES)].dropna(subset=["arrival_city"])
          .groupby(["booking_id", "arrival_city"]).size().reset_index(name="n")
          .sort_values(["booking_id", "n", "arrival_city"], ascending=[True, False, True])
          .drop_duplicates("booking_id")[["booking_id", "arrival_city"]]
          .rename(columns={"arrival_city": "destination"}))

### 5.2 Master table (fact grain: one row per fact row, all attributes joined)

In [ ]:
master = (fact
    .merge(bk_c, on="booking_id", how="left")
    .merge(pax, on="passenger_id", how="left")
    .merge(mk, on="market_id", how="left")
    .merge(pg_c, on="product_group_id", how="left")
    .merge(tr, on="transport_id", how="left")
    .merge(dest, on="booking_id", how="left"))
master["destination"] = master.destination.fillna("Unknown destination")
master["departure_year"] = master.groupby("booking_id")["first_departure_date"].transform("min").dt.year

assert len(master) == len(fact), "Joins changed the row count – check dimension keys (3.1)"
print(f"master: {len(master):,} rows × {master.shape[1]} columns")

### 5.3 Booking-level table (one row per booking)
This is the safest table for analysis: counting rows = counting bookings, and `pax` is counted once.

In [ ]:
bl = (b.reset_index()
        .merge(bk_c[["booking_id", "brand", "sales_platform", "created_by"]], on="booking_id", how="left")
        .merge(dest, on="booking_id", how="left")
        .merge(pax_b[["booking_id", "pax"]], on="booking_id", how="left")
        .merge(fact.groupby("booking_id")["market_id"].first().reset_index()
                   .merge(mk, on="market_id", how="left"), on="booking_id", how="left"))
bl["destination"] = bl.destination.fillna("Unknown destination")
bl["departure_year"] = bl.departure.dt.year
bl["margin"] = bl.sales - bl.cost
bl["lead_time_days"] = (bl.departure - bl.booking_date).dt.days
flag_cols = [c for c in bl.columns if c.startswith("flag_")]
bl["any_flag"] = bl[flag_cols].any(axis=1)

assert len(bl) == fact.booking_id.nunique(), "booking-level table must have one row per booking"
print(f"booking-level: {len(bl):,} rows")

### 5.4 Power BI table (one row per booking, 48 columns)
Adds the time, season and count columns and the status split, so the booking-level table becomes the exact table the Power BI model loads (same column names and order, so no DAX changes).

- `departure_quarter` e.g. `2022Q1`, `departure_month` 1–12, `departure_month_name` e.g. `March`
- `season`: Apr–Sep = `Summer Apr-Sep`, Oct–Mar = `Winter-sun Oct-Mar`
- `season_year`: the year the season **starts** (Jan–Mar 2023 belongs to winter 2022)
- `fact_rows`: raw fact rows per booking; `product_group_count`: distinct product groups per booking
- `sales_kept`, `cost_kept`, … : sales and cost split by status, so statuses can be shown separately
- `has_missing_cost` (renamed from `flag_missing_cost`) and `total_cost_zero`: the two cost flags, named for what they mean

In [ ]:
dep = bl["departure"]
bl["departure_quarter"]    = dep.dt.to_period("Q").astype(str).replace("NaT", np.nan)
bl["departure_month"]      = dep.dt.month.astype("Int64")
bl["departure_month_name"] = dep.dt.strftime("%B")
bl["season"]      = np.where(dep.dt.month.between(4, 9), "Summer Apr-Sep", "Winter-sun Oct-Mar")
bl["season_year"] = (dep.dt.year - dep.dt.month.le(3).astype(int)).astype("Int64")

counts = fact.groupby("booking_id").agg(fact_rows=("booking_id", "size"),
                                        product_group_count=("product_group_id", "nunique")).reset_index()
bl = bl.merge(counts, on="booking_id", how="left", validate="one_to_one")

bl = bl.rename(columns={"flag_missing_cost": "has_missing_cost"})
bl["total_cost_zero"] = bl["cost"].fillna(0).eq(0)

for name, status in [("kept", KEPT), ("cancelled", CANC), ("preliminary", PRELIM)]:
    is_s = bl["status"].eq(status)
    bl[f"sales_{name}"] = bl["sales"].where(is_s, 0)
    bl[f"cost_{name}"]  = bl["cost"].where(is_s, 0)

# Readable column order: identifiers, dimensions, measures, time, flags
first = ["booking_id", "status", "brand", "sales_platform", "sell_country", "sell_currency", "destination",
         "departure_year", "departure_quarter", "departure_month", "departure_month_name", "season", "season_year",
         "booking_date", "departure", "return_date", "cancellation_date", "lead_time_days",
         "pax", "sales", "cost", "margin", "sales_kept", "cost_kept", "sales_cancelled", "cost_cancelled",
         "sales_preliminary", "cost_preliminary", "fact_rows", "product_group_count"]
bl = bl[first + [c for c in bl.columns if c not in first]]
print(f"Power BI table: {bl.shape[0]:,} rows × {bl.shape[1]} columns")

# 6. Reconciliation
If any line says FAIL, stop and investigate before using the outputs.

In [ ]:
linked_pax = pax[pax.passenger_id.isin(fact.passenger_id)].pax.sum()
checks = {
    "master sales = raw fact sales": np.isclose(master.sales_global.sum(), fact.sales_global.sum()),
    "master cost = raw fact cost": np.isclose(master.cost_global.sum(), fact.cost_global.sum()),
    "one row per booking": bl.booking_id.is_unique and len(bl) == fact.booking_id.nunique(),
    "booking-level sales = raw fact sales": np.isclose(bl.sales.sum(), fact.sales_global.sum()),
    "booking-level cost = raw fact cost": np.isclose(bl.cost.sum(), fact.cost_global.sum()),
    "status counts sum to unique bookings": bl.status.value_counts().sum() == fact.booking_id.nunique(),
    "passengers: booking-level = dim_passenger (linked)": bl.pax.sum() == linked_pax,
    "fact_rows sum = raw rows": bl.fact_rows.sum() == len(fact),
    "season filled for every booking": bl.season.notna().all() and bl.departure.notna().all(),
    "sales split adds up to sales": np.allclose(bl.sales_kept + bl.sales_cancelled + bl.sales_preliminary,
                                                bl.sales.fillna(0)),
    "cost split adds up to cost": np.allclose((bl.cost_kept + bl.cost_cancelled + bl.cost_preliminary).fillna(0),
                                              bl.cost.fillna(0)),
    "margin = sales - cost": np.allclose(bl.margin.fillna(0), bl.sales.fillna(0) - bl.cost.fillna(0)),
    "no empty brand": bl.brand.notna().all(),
    "Power BI table has 48 columns": bl.shape[1] == 48,
}
for k, v in checks.items():
    print(f"{'PASS' if v else 'FAIL'}  {k}")
    log("Reconciliation", k, 0 if v else 1, "Must PASS")
print("\nStatus counts:", bl.status.value_counts().to_dict())
print(f"Total sales (SEK): {fact.sales_global.sum():,.0f} | Total cost (SEK): {fact.cost_global.sum():,.0f}")

# 7. Export

In [ ]:
master.to_csv(OUT / "master_data.csv", index=False)
pd.DataFrame(quality_log).to_csv(OUT / "data_quality_log.csv", index=False)
bl[bl.any_flag].to_csv(OUT / "booking_exceptions.csv", index=False)

# The Power BI table is only saved if every reconciliation check passed
if all(checks.values()):
    bl.to_csv(OUT / "booking_level_powerbi.csv", index=False)
    print(f"Saved booking_level_powerbi.csv: {bl.shape[0]:,} rows × {bl.shape[1]} columns")
else:
    print("booking_level_powerbi.csv NOT saved – fix the FAIL lines in section 6 first.")
for p in sorted(OUT.glob("*.csv")):
    print(p)

### 7.1 Known-answer test on the saved file
Reads the exact file Power BI loads and recomputes one dashboard figure (kept bookings, sales and margin % for the largest brand in `YEAR`), then checks it against the same figure calculated independently from the raw input tables.

In [ ]:
chk = pd.read_csv(OUT / "booking_level_powerbi.csv")
BRAND = chk[chk.status == KEPT].brand.value_counts().idxmax()
gp = chk[(chk.brand == BRAND) & (chk.departure_year == YEAR) & (chk.status == KEPT)]
from_file = (len(gp), round(gp.sales.sum()), round((gp.sales.sum() - gp.cost.sum()) / gp.sales.sum() * 100, 1))

raw_k = fact.merge(bk[["booking_id", "booking_status", "brand"]], on="booking_id")
raw_k = raw_k.merge(fact.groupby("booking_id").first_departure_date.min().dt.year.rename("dep_year"), on="booking_id")
r = raw_k[(raw_k.brand == BRAND) & (raw_k.dep_year == YEAR) & (raw_k.booking_status == KEPT)]
from_raw = (r.booking_id.nunique(), round(r.sales_global.sum()),
            round((r.sales_global.sum() - r.cost_global.sum()) / r.sales_global.sum() * 100, 1))

print(f"{BRAND}, {YEAR}, kept bookings")
print(f"  Saved file: {from_file[0]:,} bookings | {from_file[1]:,} SEK | {from_file[2]} % margin")
print(f"  Raw tables: {from_raw[0]:,} bookings | {from_raw[1]:,} SEK | {from_raw[2]} % margin")
print("PASS  saved file matches the raw data" if from_file == from_raw else "CHECK  saved file differs from the raw data")

# 8. Findings
All analysis uses the **booking-level table** (`bl`), so bookings and passengers are never double counted.

Definitions used below:
- **Kept booking** = status `Booking`. **Cancelled** = status `Cancellation`. Preliminary excluded.
- **Total bookings (demand incl. cancelled)** = kept + cancelled.
- **Cancellation rate** = cancelled ÷ (kept + cancelled).
- **Kept sales / cost / margin** = sums over kept bookings, SEK. **Margin %** = Σ margin ÷ Σ sales.
- **Cancelled value** = |Σ sales of cancelled bookings| (subject to check 3.6).

In [ ]:
def summarise(df, by):
    x = df[df.status.isin([KEPT, CANC])].copy()
    x["is_kept"] = x.status.eq(KEPT)
    x["is_canc"] = x.status.eq(CANC)
    x["kept_sales"] = x.sales.where(x.is_kept, 0)
    x["kept_cost"] = x.cost.where(x.is_kept, 0)
    x["canc_sales"] = x.sales.where(x.is_canc, 0)
    g = x.groupby(by)
    out = pd.DataFrame({
        "kept_bookings": g.is_kept.sum(),
        "cancelled_bookings": g.is_canc.sum(),
        "passengers_incl_canc": g.pax.sum(),
        "kept_sales": g.kept_sales.sum(),
        "kept_cost": g.kept_cost.sum(),
        "cancelled_value": g.canc_sales.sum().abs(),
    })
    out["total_bookings"] = out.kept_bookings + out.cancelled_bookings
    out["cancellation_rate"] = out.cancelled_bookings / out.total_bookings
    out["kept_margin"] = out.kept_sales - out.kept_cost
    out["margin_pct"] = out.kept_margin / out.kept_sales.replace(0, np.nan)
    return out

### 8.1 Context: the COVID effect by year

In [ ]:
by_year = summarise(bl, "departure_year")
display(by_year[["total_bookings", "kept_bookings", "cancelled_bookings", "cancellation_rate",
                 "kept_sales", "kept_margin", "margin_pct"]])
ax = by_year[["kept_bookings", "cancelled_bookings"]].plot(kind="bar", stacked=True, figsize=(7, 4))
ax.set_title("Bookings by departure year and status"); ax.set_ylabel("Bookings"); plt.tight_layout(); plt.show()

### 8.2 Q1 – Demand: brands and destinations (incl. cancelled bookings)

In [ ]:
q1_brand = summarise(bl, ["brand", "departure_year"])["total_bookings"].unstack()
display(q1_brand)

q1_dest = summarise(bl, "destination").sort_values("total_bookings", ascending=False)
print("Most bookings:"); display(q1_dest[["total_bookings", "passengers_incl_canc"]].head(10))
print("Fewest bookings:"); display(q1_dest[["total_bookings", "passengers_incl_canc"]].tail(10))

### 8.3 Q2 – Cancellations
Compared **within each year**, so the COVID level does not dominate. `rate_vs_year` > 0 means above that year's overall rate.

In [ ]:
q2 = summarise(bl, ["departure_year", "destination"])
q2 = q2[q2.total_bookings >= MIN_BOOKINGS].copy()
q2["rate_vs_year"] = q2.cancellation_rate - q2.index.get_level_values(0).map(by_year.cancellation_rate)
display(q2.sort_values("rate_vs_year", ascending=False)
          [["total_bookings", "cancellation_rate", "rate_vs_year", "cancelled_value"]].head(15))
display(summarise(bl, ["brand", "departure_year"])["cancellation_rate"].unstack())

### 8.4 Q3 – Sales and margin from kept bookings, and channels

In [ ]:
q3_brand = summarise(bl[bl.departure_year == YEAR], "brand")
display(q3_brand[["kept_bookings", "kept_sales", "kept_margin", "margin_pct"]]
          .sort_values("kept_margin", ascending=False))

q3_dest = summarise(bl[bl.departure_year == YEAR], "destination")
q3_dest = q3_dest[q3_dest.kept_bookings >= MIN_BOOKINGS]
print(f"Destinations with >= {MIN_BOOKINGS} kept bookings in {YEAR}, by margin:")
display(q3_dest[["kept_bookings", "kept_sales", "kept_margin", "margin_pct"]]
          .sort_values("kept_margin", ascending=False))

In [ ]:
ch = summarise(bl[bl.departure_year == YEAR], ["brand", "sales_platform"])
print("Share of kept bookings by channel, per brand:")
display(ch.kept_bookings.unstack().pipe(lambda t: t.div(t.sum(axis=1), axis=0)))
print("Margin % by brand and channel:")
display(ch.margin_pct.unstack())

### 8.5 Decision rule (main question)
Same rule as the Power BI `Outcome` measure – this is the independent calculation to compare against.

In [ ]:
def decision_table(year, min_bookings):
    d = summarise(bl[bl.departure_year == year], "destination")
    overall = summarise(bl[bl.departure_year == year].assign(all="all"), "all").iloc[0]
    median_demand = d[(d.index != "Unknown destination") &
                      (d.kept_bookings >= min_bookings)].kept_bookings.median()

    def classify(r):
        if r.name == "Unknown destination": return "Not classified (unknown)"
        if r.kept_bookings < min_bookings:  return "Too small to classify"
        hi_d, hi_m = r.kept_bookings > median_demand, r.margin_pct > overall.margin_pct
        flag = r.cancellation_rate > overall.cancellation_rate
        if hi_d and hi_m: return "Investigate first (high cancellations)" if flag else "Consider more resources"
        if hi_d: return "Review plans (pricing/cost)"
        if hi_m: return "Investigate"
        return "Review plans (scale back?)"

    d["outcome"] = d.apply(classify, axis=1)
    d["margin_gap_pp"] = (d.margin_pct - overall.margin_pct) * 100
    return d, overall, median_demand

d, overall, median_demand = decision_table(YEAR, MIN_BOOKINGS)
print(f"{YEAR}: overall margin % {overall.margin_pct:.2%} | overall cancellation rate "
      f"{overall.cancellation_rate:.2%} | median kept bookings {median_demand}")
display(d.outcome.value_counts())
display(d.sort_values("kept_bookings", ascending=False)
          [["kept_bookings", "cancellation_rate", "kept_sales", "kept_margin",
            "margin_pct", "margin_gap_pp", "outcome"]])

In [ ]:
plot_d = d[d.kept_bookings >= MIN_BOOKINGS].drop(index="Unknown destination", errors="ignore")
fig, ax = plt.subplots(figsize=(8, 5))
for outcome, grp in plot_d.groupby("outcome"):
    ax.scatter(grp.kept_bookings, grp.margin_pct, s=grp.kept_sales / plot_d.kept_sales.max() * 600 + 20,
               alpha=.6, label=outcome)
ax.axvline(median_demand, ls="--", c="grey"); ax.axhline(overall.margin_pct, ls="--", c="grey")
ax.set_xlabel("Kept bookings"); ax.set_ylabel("Margin %"); ax.set_title(f"Destinations {YEAR}: demand vs margin %")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
if len(plot_d): ax.legend(fontsize=8)
else: print(f"No destination has >= {MIN_BOOKINGS} kept bookings in {YEAR} – lower MIN_BOOKINGS or check the data")
plt.tight_layout(); plt.show() 

### 8.6 Robustness: does the result depend on our threshold choices?
If only a few destinations change outcome, the finding holds under reasonable alternatives.

In [ ]:
for m in [20, 30, 50]:
    alt, _, _ = decision_table(YEAR, m)
    print(f"MIN_BOOKINGS = {m}: {(alt.outcome != d.outcome).sum()} of {len(d)} destinations change outcome")

### 8.7 Stability: same destinations, previous year
A destination in the same outcome two years running is stronger evidence than one year alone (remember COVID in earlier years).

In [ ]:
prev, _, _ = decision_table(YEAR - 1, MIN_BOOKINGS)
both = d[["outcome"]].join(prev[["outcome"]], how="inner",
                           lsuffix=f"_{YEAR}", rsuffix=f"_{YEAR-1}")

print(f"All destinations present in both years ({len(both)}):")
display(pd.crosstab(both[f"outcome_{YEAR-1}"], both[f"outcome_{YEAR}"]))

not_classified = ["Too small to classify", "Not classified (unknown)"]
classified = both[~both[f"outcome_{YEAR}"].isin(not_classified) &
                  ~both[f"outcome_{YEAR-1}"].isin(not_classified)]
same = classified[classified[f"outcome_{YEAR}"] == classified[f"outcome_{YEAR-1}"]]

print(f"\n{len(same)} of {len(classified)} destinations classified in both years "
      f"have the same outcome in {YEAR-1} and {YEAR}")
print("\nSame outcome both years:")
display(same)
print("Changed outcome:")
display(classified[classified[f"outcome_{YEAR}"] != classified[f"outcome_{YEAR-1}"]])

### 8.8 Booking timing (lead time)
Days from first booking date to first departure, kept bookings. **Caution:** `booking_date` may be recorded per row, so the earliest date per booking is used.

In [ ]:
lt = bl[(bl.status == KEPT) & (bl.lead_time_days >= 0)]
print("Median lead time (days) by departure year and brand:")
display(lt.pivot_table(index="brand", columns="departure_year", values="lead_time_days", aggfunc="median"))

### 8.9 Trace one result back to the source
Dashboard figure → findings table → bookings → raw rows. Use this for the presentation.

In [ ]:
known = d.drop(index="Unknown destination", errors="ignore")
TRACE = known.kept_bookings.idxmax()          # or type a destination name, e.g. "Malaga"
rows = bl[(bl.departure_year == YEAR) & (bl.destination == TRACE) & (bl.status == KEPT)]
print(f"Trace: {TRACE}, {YEAR}")
print(f"  Kept bookings: {len(rows)}  (findings table: {int(d.loc[TRACE, 'kept_bookings'])})")
print(f"  Kept sales:    {rows.sales.sum():,.0f} SEK  (findings table: {d.loc[TRACE, 'kept_sales']:,.0f})")
print(f"  Margin %:      {(rows.sales.sum() - rows.cost.sum()) / rows.sales.sum():.2%}  "
      f"(findings table: {d.loc[TRACE, 'margin_pct']:.2%})")
one = rows.booking_id.iloc[0]
print(f"\nOne booking ({one}) at source-row level:")
display(fact[fact.booking_id == one][["booking_id", "product_group_id", "transport_id",
                                      "booking_date", "first_departure_date", "sales_global", "cost_global"]])

### 8.10 Export findings

In [ ]:
by_year.to_csv(OUT / "findings_by_year.csv")
q1_dest.to_csv(OUT / "findings_q1_destinations.csv")
q2.to_csv(OUT / "findings_q2_cancellations.csv")
q3_dest.to_csv(OUT / "findings_q3_destinations.csv")
d.to_csv(OUT / f"validation_destinations_{YEAR}.csv")
pd.DataFrame(quality_log).to_csv(OUT / "data_quality_log.csv", index=False)
print("Exported. Compare validation_destinations_%d.csv with Power BI (year = %d, no other filters)." % (YEAR, YEAR))

# 9. From output to finding
Fill in **only from the outputs above**. Write two or three findings, not ten.

| | Finding 1 | Finding 2 |
|---|---|---|
| **Observation** – what the data shows (number, period, definition) | | |
| **Interpretation** – what it suggests | | |
| **Implication** – why it matters to the Commercial Manager | | |
| **Recommended action** – what they could consider | | |
| **Limitation** – what the data cannot tell us | | |
| **Evidence** – which cell / output file | | |

Language to use: *"The data shows…", "This suggests…", "We recommend considering…", "This would need to be tested…"*

In [ ]:
# 1. Source-to-dashboard: kept sales in YEAR straight from the raw files
dep = fact.groupby("booking_id").first_departure_date.min().dt.year
st = bk.set_index("booking_id").booking_status
ids = dep[dep == YEAR].index.intersection(st[st == KEPT].index)
print(f"Raw kept sales {YEAR}: {fact[fact.booking_id.isin(ids)].sales_global.sum():,.2f} SEK  (compare with Power BI, year {YEAR})")

# 2. Missing and zero cost by destination, kept bookings in YEAR
k = bl[(bl.departure_year == YEAR) & (bl.status == KEPT)]
mc = k.groupby("destination").agg(kept=("booking_id", "size"), missing_cost=("has_missing_cost", "sum"),
                                  zero_cost=("cost", lambda s: (s.fillna(0) == 0).sum()))
mc["missing_share"] = mc.missing_cost / mc.kept
display(mc[mc.kept >= MIN_BOOKINGS].sort_values("missing_share", ascending=False).head(10))

# 3. Why destinations are unknown: no flight arrangement?
u = fact[fact.booking_id.isin(bl[bl.destination == "Unknown destination"].booking_id)]
print(f"Unknown-destination rows with no transport_id: {u.transport_id.isna().mean():.0%}")

# 4. How close is the destination nearest to the cancellation threshold?
classified = d[~d.outcome.isin(["Too small to classify", "Not classified (unknown)"])]
nearest = (classified.cancellation_rate - overall.cancellation_rate).abs().idxmin()
print(f"{nearest} cancellation rate: {d.loc[nearest, 'cancellation_rate']:.2%} vs overall {overall.cancellation_rate:.2%}")

In [ ]:
# Does missing or zero cost inflate the margin? Top 4 destinations by kept sales in YEAR
k = bl[(bl.departure_year == YEAR) & (bl.status == KEPT)]
top4 = k.groupby("destination").sales.sum().drop("Unknown destination", errors="ignore").nlargest(4).index
overall_pct = k.margin.sum() / k.sales.sum()
for dest in top4:
    x = k[k.destination == dest]
    y = x[~x.has_missing_cost & (x.cost.fillna(0) != 0)]
    print(f"{dest:12} margin {x.margin.sum()/x.sales.sum():.2%} -> without missing/zero-cost bookings "
          f"{y.margin.sum()/y.sales.sum():.2%}  (overall {overall_pct:.2%})")